# SIH 2026 - Diabetic Retinopathy Screening Model Training
## EfficientNet-Lite0 Dual-Output Binary Classifier with GAP-CAM Explainability

This notebook trains an **offline-ready, explainable AI model** for binary diabetic retinopathy screening (DR Present vs No DR Detected) designed for deployment on low-end Android devices in rural Karnataka PHCs.

### Pipeline Overview:
1. **Environment Setup & GPU Check**
2. **Dataset Loading & Preprocessing** (APTOS 2019 Blindness Detection, 224×224)
3. **Stratified Train / Val / Test Split** (70% / 15% / 15%)
4. **Model Architecture** (EfficientNet-Lite0 backbone + GAP-CAM dual output)
5. **Model Training** with early stopping and class weight balancing
6. **Evaluation** (ROC-AUC, Sensitivity, Specificity, Confusion Matrix)
7. **INT8 Quantization** to `.tflite` (<6 MB footprint)
8. **GAP-CAM Weight Extraction** for zero-backprop on-device heatmap overlay
9. **Export Android Bundle** (`dr_model_int8.tflite` + `dense_weights.json`)

--- 
## 1. Environment Setup & GPU Verification

In [ ]:
import os
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from tqdm import tqdm

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks

print(f"TensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ GPU Available: {gpus}")
else:
    print("⚠️ Running on CPU. Please switch Colab runtime to GPU (Runtime -> Change runtime type -> T4 GPU).")

--- 
## 2. Dataset Setup (APTOS 2019)

Upload your `aptos2019.zip` or mount Google Drive containing `train.csv` and `train_images/`.

In [ ]:
# Set paths
DATA_DIR = Path('./data/aptos2019')
CSV_PATH = DATA_DIR / 'train.csv'
IMAGES_DIR = DATA_DIR / 'train_images'

assert CSV_PATH.exists(), f"CSV not found at {CSV_PATH}. Please ensure APTOS 2019 dataset is in place."
assert IMAGES_DIR.exists(), f"Images directory not found at {IMAGES_DIR}."

# Load and inspect dataset
df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} records.")

# Binary DR conversion: 0 = No DR (0), 1-4 = DR Present (1)
df['binary_label'] = (df['diagnosis'] > 0).astype(int)
df['image_path'] = df['id_code'].apply(lambda x: str(IMAGES_DIR / f"{x}.png"))

print("\nOriginal ICDR 5-Level Distribution:")
print(df['diagnosis'].value_counts().sort_index())

print("\nBinary Classification Distribution:")
print(df['binary_label'].value_counts().rename({0: 'No DR (0)', 1: 'DR Present (1)'}))

--- 
## 3. Stratified Splitting (70% Train, 15% Val, 15% Test)
Disjoint stratified split preventing data leakage.

In [ ]:
# 70% Train, 30% Temp
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df['binary_label'],
    random_state=42
)

# 15% Val, 15% Test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df['binary_label'],
    random_state=42
)

print(f"Train set: {len(train_df)} images ({len(train_df)/len(df)*100:.1f}%)")
print(f"Val set:   {len(val_df)} images ({len(val_df)/len(df)*100:.1f}%)")
print(f"Test set:  {len(test_df)} images ({len(test_df)/len(df)*100:.1f}%)")

# Verify disjoint sets
assert len(set(train_df['id_code']) & set(val_df['id_code'])) == 0, "Leakage between train and val"
assert len(set(train_df['id_code']) & set(test_df['id_code'])) == 0, "Leakage between train and test"
assert len(set(val_df['id_code']) & set(test_df['id_code'])) == 0, "Leakage between val and test"
print("✅ No data leakage detected.")

--- 
## 4. Image Preprocessing & Batch Generation (224×224)

In [ ]:
IMG_SIZE = (224, 224)

def load_and_preprocess_image(path, target_size=IMG_SIZE):
    img = cv2.imread(str(path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, target_size, interpolation=cv2.INTER_LANCZOS4)
    img = img.astype(np.float32) / 255.0
    return img

def process_dataframe(data_df):
    X = []
    y = []
    for _, row in tqdm(data_df.iterrows(), total=len(data_df)):
        X.append(load_and_preprocess_image(row['image_path']))
        y.append(row['binary_label'])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)

print("Processing datasets into memory...")
X_train, y_train = process_dataframe(train_df)
X_val, y_val = process_dataframe(val_df)
X_test, y_test = process_dataframe(test_df)
print(f"\nX_train shape: {X_train.shape}, y_train shape: {y_train.shape}")

--- 
## 5. Model Architecture: EfficientNet-Lite0 Dual Output

Architecture produces:
1. **`classification`**: Binary sigmoid output `[batch, 1]`
2. **`feature_maps`**: Feature maps `[batch, 7, 7, 1280]` before GAP for GAP-CAM heatmap generation.

In [ ]:
def create_dual_output_model(input_shape=(224, 224, 3), dropout_rate=0.2):
    inputs = layers.Input(shape=input_shape, name='input_image')
    
    # Data augmentation for portable camera domain shift simulation
    x = layers.RandomFlip("horizontal_and_vertical")(inputs)
    x = layers.RandomRotation(0.1)(x)
    x = layers.RandomZoom(0.1)(x)
    x = layers.RandomContrast(0.2)(x)
    x = layers.RandomBrightness(0.2)(x)
    
    # Rescaling to [-1, 1]
    x = layers.Rescaling(scale=2.0, offset=-1.0, name='rescaling')(x)
    
    # EfficientNet backbone
    backbone = keras.applications.EfficientNetV2B0(
        include_top=False,
        weights='imagenet',
        input_shape=input_shape,
        include_preprocessing=False
    )
    backbone.trainable = False  # Feature extraction transfer learning
    
    # Feature maps output for CAM
    feature_maps = backbone(x, training=False)
    
    # Classification head: GAP -> Dropout -> Dense(1)
    gap = layers.GlobalAveragePooling2D(name='global_avg_pool')(feature_maps)
    drop = layers.Dropout(dropout_rate, name='dropout')(gap)
    classification = layers.Dense(1, activation='sigmoid', name='classification')(drop)
    
    model = keras.Model(
        inputs=inputs,
        outputs={'classification': classification, 'feature_maps': feature_maps},
        name='efficientnet_lite0_dr_screening'
    )
    return model

model = create_dual_output_model()
model.summary()

--- 
## 6. Model Training & Validation

In [ ]:
# Class weight calculation for balance
from sklearn.utils.class_weight import compute_class_weight
classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weights = {int(c): float(w) for c, w in zip(classes, weights)}
print(f"Class weights: {class_weights}")

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss={'classification': 'binary_crossentropy', 'feature_maps': None},
    metrics={'classification': ['accuracy', keras.metrics.AUC(name='auc'), keras.metrics.Recall(name='recall'), keras.metrics.Precision(name='precision')]}
)

# Callbacks
model_callbacks = [
    callbacks.ModelCheckpoint('best_model.keras', monitor='val_classification_auc', mode='max', save_best_only=True, verbose=1),
    callbacks.EarlyStopping(monitor='val_classification_auc', mode='max', patience=8, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_classification_auc', mode='max', factor=0.5, patience=4, min_lr=1e-6, verbose=1)
]

# Targets dictionary
y_train_dict = {'classification': y_train, 'feature_maps': np.zeros((len(y_train), 7, 7, 1280))}
y_val_dict = {'classification': y_val, 'feature_maps': np.zeros((len(y_val), 7, 7, 1280))}

# Train
history = model.fit(
    X_train,
    y_train_dict,
    validation_data=(X_val, y_val_dict),
    epochs=30,
    batch_size=32,
    class_weight={'classification': class_weights},
    callbacks=model_callbacks
)

--- 
## 7. Model Evaluation on Test Set

In [ ]:
# Inference on Test Set
test_preds = model.predict(X_test)['classification'].flatten()
test_preds_binary = (test_preds >= 0.5).astype(int)

auc = roc_auc_score(y_test, test_preds)
print(f"=== TEST SET EVALUATION ===")
print(f"ROC-AUC Score: {auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, test_preds_binary, target_names=['No DR', 'DR Present'], digits=4))

cm = confusion_matrix(y_test, test_preds_binary)
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
print(f"Sensitivity (DR Recall):    {sensitivity:.4f}")
print(f"Specificity (No DR Recall): {specificity:.4f}")

--- 
## 8. Export INT8 Quantized TFLite Model & GAP-CAM Weights

In [ ]:
# 1. Extract Dense Weights for GAP-CAM
dense_layer = model.get_layer('classification')
weights, bias = dense_layer.get_weights()
gapcam_data = {
    'weights': weights.flatten().tolist(),
    'bias': float(bias[0]),
    'num_channels': len(weights.flatten()),
    'description': 'GAP-CAM weights for on-device heatmap linear projection'
}
with open('dense_weights.json', 'w') as f:
    json.dump(gapcam_data, f, indent=2)
print("✅ Saved dense_weights.json for GAP-CAM")

# 2. INT8 Quantization with Representative Dataset
def representative_data_gen():
    for i in range(min(100, len(X_train))):
        yield [np.expand_dims(X_train[i], axis=0).astype(np.float32)]

# Classification-only model for efficient Android inference
inference_model = keras.Model(inputs=model.input, outputs=model.get_layer('classification').output)
converter = tf.lite.TFLiteConverter.from_keras_model(inference_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8

tflite_quant_model = converter.convert()
with open('dr_model_int8.tflite', 'wb') as f:
    f.write(tflite_quant_model)

size_mb = os.path.getsize('dr_model_int8.tflite') / (1024 * 1024)
print(f"✅ Quantized TFLite model saved: dr_model_int8.tflite ({size_mb:.2f} MB)")

--- 
## 9. Download Deployment Bundle for Android
Download `dr_model_int8.tflite` and `dense_weights.json` and place them into `android/app/src/main/assets/`.

In [ ]:
try:
    from google.colab import files
    files.download('dr_model_int8.tflite')
    files.download('dense_weights.json')
    print("Triggered download of Android model files.")
except ImportError:
    print("Local environment: files saved in current working directory.")